In [10]:
import pandas as pd
from tqdm import tqdm

In [11]:
filename = "4square/Foursquare.h5"

In [13]:
print("Reading csv files")
trip_df = pd.read_csv(f"4square/Foursquare-POIs.csv")
trip_df = trip_df.loc[:, ~trip_df.columns.str.contains("POI")]
poi = pd.read_csv(f"4square/Foursquare-POIs.csv")

Reading csv files


In [14]:
# Write to an HDF5

with pd.HDFStore(filename) as store:
    store.put('pois', poi, format="table")
    store.put('trips', trip_df)

tqdm.write(f"✅ Done writing to {filename}")

✅ Done writing to 4square/Foursquare.h5


In [15]:
trip_df = pd.read_hdf(filename, key='trips')
trip_df.head(10) #type: ignore

,traj_id,user_id,trip,seq_i,time,lng,lat,timestamp,delta_t,venue_id,venue_category_name,radius_used(m)
0,0,0,0,0,2012-04-04 09:10:41+00:00,139.728553,35.619699,1333530641,0,4b22504cf964a520704524e3,Train Station,250
1,0,0,0,1,2012-04-04 09:13:57+00:00,139.727576,35.618372,1333530837,196,4b53b9c4f964a5205ca927e3,Building,250
2,0,0,0,2,2012-04-04 10:18:11+00:00,139.731144,35.619121,1333534691,4050,4c985494d799a1cd7189b152,Bakery,250
3,0,0,0,3,2012-04-04 12:11:50+00:00,139.740429,35.630146,1333541510,10869,4b0e60adf964a520305723e3,Train Station,250
4,0,0,0,4,2012-04-04 12:22:29+00:00,139.715953,35.562387,1333542149,11508,4b2837d8f964a5200e9124e3,Train Station,250
5,0,0,0,5,2012-04-04 12:25:22+00:00,139.715070,35.563292,1333542322,11681,4be12c19c1732d7ff9675b9a,Arcade,250
6,0,0,0,6,2012-04-04 14:56:25+00:00,139.716697,35.564650,1333551385,20744,4e7eedae9adf979a3f17456f,Parking,250
7,1,0,1,0,2012-04-08 01:56:30+00:00,139.715070,35.563292,1333850190,0,4be12c19c1732d7ff9675b9a,Arcade,250
8,1,0,1,1,2012-04-08 02:25:10+00:00,139.715953,35.562387,1333851910,1720,4b2837d8f964a5200e9124e3,Train Station,250
9,1,0,1,2,2012-04-08 02:52:15+00:00,139.773018,35.698596,1333853535,3345,4b19f917f964a520abe623e3,Train Station,250


In [16]:
poi_df = pd.read_hdf(filename, key='pois')
poi_df.head() #type: ignore

,traj_id,user_id,trip,seq_i,time,lng,lat,timestamp,delta_t,venue_id,...,POI_6_dist(m),POI_6_dir(deg),POI_7_dist(m),POI_7_dir(deg),POI_8_dist(m),POI_8_dir(deg),POI_9_dist(m),POI_9_dir(deg),POI_10_dist(m),POI_10_dir(deg)
0,0,0,0,0,2012-04-04 09:10:41+00:00,139.728553,35.619699,1333530641,0,4b22504cf964a520704524e3,...,156.246853,76.228108,163.168947,197.257203,176.414885,224.393845,199.177811,357.792252,201.404647,91.650519
1,0,0,0,1,2012-04-04 09:13:57+00:00,139.727576,35.618372,1333530837,196,4b53b9c4f964a5205ca927e3,...,223.427041,178.833674,234.889588,39.224162,236.416112,35.463460,238.159023,37.805738,NaN,NaN
2,0,0,0,2,2012-04-04 10:18:11+00:00,139.731144,35.619121,1333534691,4050,4c985494d799a1cd7189b152,...,205.582372,300.759029,215.425893,300.572803,242.594484,277.106803,243.132200,305.283718,NaN,NaN
3,0,0,0,3,2012-04-04 12:11:50+00:00,139.740429,35.630146,1333541510,10869,4b0e60adf964a520305723e3,...,216.112476,210.691461,226.332889,231.309888,239.281370,176.898014,241.891420,274.068568,NaN,NaN
4,0,0,0,4,2012-04-04 12:22:29+00:00,139.715953,35.562387,1333542149,11508,4b2837d8f964a5200e9124e3,...,156.949384,216.278529,169.344050,127.553655,185.310612,149.651210,205.758378,72.401719,245.232260,327.032069


In [17]:
def get_utm_zones_from_df(df, lat_col="lat", lon_col="lng", debug=True):
    import numpy as np
    
    # Bounding box
    min_lat, max_lat = df[lat_col].min(), df[lat_col].max()
    min_lon, max_lon = df[lon_col].min(), df[lon_col].max()
    
    # Print bounding box if debugging
    if debug:
        print(f"Latitude range: {min_lat} → {max_lat}")
        print(f"Longitude range: {min_lon} → {max_lon}")
    
    # 4 corners of bounding box
    corners = [
        (min_lat, min_lon),
        (min_lat, max_lon),
        (max_lat, min_lon),
        (max_lat, max_lon)
    ]
    
    zones = set()
    for lat, lon in corners:
        zone_number = int((lon + 180) // 6) + 1
        hemisphere = "N" if lat >= 0 else "S"
        zones.add((zone_number, hemisphere))
    
    return sorted(zones)

def get_utm_zone_from_center(df, lat_col="lat", lon_col="lng"):
    center_lat = (df[lat_col].min() + df[lat_col].max()) / 2
    center_lon = (df[lon_col].min() + df[lon_col].max()) / 2
    zone_number = int((center_lon + 180) // 6) + 1
    hemisphere = "N" if center_lat >= 0 else "S"
    return (zone_number, hemisphere)


zones1 = get_utm_zone_from_center(trip_df, lat_col="lat", lon_col="lng")
zones2 = get_utm_zones_from_df(trip_df, lat_col="lat", lon_col="lng")
print("UTM Zones:", zones1)
print("UTM Zones:", zones2)

Latitude range: 35.51105296361414 → 35.867150422391695
Longitude range: 139.47176970135368 → 139.90705847740173
UTM Zones: (54, 'N')
UTM Zones: [(54, 'N')]
